In [2]:

import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import QuantileTransformer, LabelEncoder
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import lightgbm as lgb
import pickle

class Config:
    """경로 자동 보정 기능이 추가된 설정 클래스"""
    SEED = 42
    SEEDS = [41, 42, 43]
    
    import os
    
    # -----------------------------------------------------------
    # [핵심 수정] 파일 위치 자동 탐색 로직
    # -----------------------------------------------------------
    # 1. 현재 실행 위치(CWD) 확인
    current_path = os.getcwd()
    print(f"📍 현재 파이썬 실행 위치: {current_path}")
    
    # 2. 파일이 있을만한 후보 경로들을 모두 검사
    target_file = "train_features_dl.csv"
    
    # 후보 1: 현재 폴더 바로 안에 파일이 있는 경우 (스크립트가 data 폴더 안에 있을 때)
    if os.path.exists(os.path.join(current_path, target_file)):
        DATA_DIR = current_path
        print(f"✅ 파일이 현재 폴더에 있습니다. 경로를 '.'로 설정합니다.")
        
    # 후보 2: 현재 폴더 안에 'data' 폴더가 있고 그 안에 파일이 있는 경우 (일반적인 경우)
    elif os.path.exists(os.path.join(current_path, "data", target_file)):
        DATA_DIR = os.path.join(current_path, "data")
        print(f"✅ 'data' 폴더 안에서 파일을 찾았습니다.")
        
    # 후보 3: 한 단계 상위 폴더의 'data' 폴더에 있는 경우
    elif os.path.exists(os.path.join(os.path.dirname(current_path), "data", target_file)):
        DATA_DIR = os.path.join(os.path.dirname(current_path), "data")
        print(f"✅ 상위 폴더의 'data' 폴더에서 파일을 찾았습니다.")
        
    else:
        # 못 찾았을 경우 절대 경로를 직접 입력해야 함
        print(f"❌ '{target_file}' 파일을 자동으로 찾지 못했습니다.")
        print(f"   현재 위치의 파일 목록: {os.listdir(current_path)}")
        # 비상용 기본값 (에러가 날 수 있음)
        DATA_DIR = "./data"

    OUTPUT_DIR = "./output"
    
    # 경로 조립
    TRAIN_FEATURE_PATH = os.path.join(DATA_DIR, "train_features_dl.csv")
    TEST_FEATURE_PATH = os.path.join(DATA_DIR, "test_features_dl.csv")
    
    # 학습 설정
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    BATCH_SIZE = 512
    EPOCHS = 60
    LR = 5e-4
    WEIGHT_DECAY = 1e-4

cfg = Config()

# 출력 폴더가 없으면 생성
if not os.path.exists(cfg.OUTPUT_DIR):
    os.makedirs(cfg.OUTPUT_DIR)
    print(f"📁 Created output directory: {cfg.OUTPUT_DIR}")


# ============================================================================
# 유틸리티 함수
# ============================================================================
def seed_everything(seed=42):
    """재현성 확보를 위한 시드 고정"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(cfg.SEED)


# ============================================================================
# 데이터 로드 및 전처리
# ============================================================================
print(f"📂 Loading data from {cfg.DATA_DIR}...")

# 데이터 파일 존재 여부 확인
if not os.path.exists(cfg.TRAIN_FEATURE_PATH):
    raise FileNotFoundError(
        f"❌ 학습 데이터를 찾을 수 없습니다!\n"
        f"   확인 경로: {cfg.TRAIN_FEATURE_PATH}\n"
        f"   'data' 폴더 안에 'train_features_dl.csv' 파일이 있는지 확인해주세요."
    )

if not os.path.exists(cfg.TEST_FEATURE_PATH):
    raise FileNotFoundError(
        f"❌ 테스트 데이터를 찾을 수 없습니다!\n"
        f"   확인 경로: {cfg.TEST_FEATURE_PATH}\n"
        f"   'data' 폴더 안에 'test_features_dl.csv' 파일이 있는지 확인해주세요."
    )

df = pd.read_csv(cfg.TRAIN_FEATURE_PATH)
test_df = pd.read_csv(cfg.TEST_FEATURE_PATH)
print(f"✅ Data loaded successfully!")
print(f"   Train shape: {df.shape}")
print(f"   Test shape: {test_df.shape}")

# Label Encoding: 품목 ID를 정수 인덱스로 변환
le_lead = LabelEncoder()
le_follow = LabelEncoder()

# Train + Test 전체에서 유니크한 품목 ID 추출 (일관성 확보)
all_leads = pd.concat([df['leading_item_id'], test_df['leading_item_id']]).unique()
all_follows = pd.concat([df['following_item_id'], test_df['following_item_id']]).unique()

le_lead.fit(all_leads)
le_follow.fit(all_follows)

# 인덱스 변환 적용
df['lead_idx'] = le_lead.transform(df['leading_item_id'])
df['follow_idx'] = le_follow.transform(df['following_item_id'])
test_df['lead_idx'] = le_lead.transform(test_df['leading_item_id'])
test_df['follow_idx'] = le_follow.transform(test_df['following_item_id'])

n_leads = len(le_lead.classes_)
n_follows = len(le_follow.classes_)

# 피처 목록 정의
feature_cols = [
    # 선행 품목(b) 관련 피처
    "b_t", "delta_b", "b_ma3", "b_ma6", "b_ma12", "b_ratio_3",
    
    # 후행 품목(a) 관련 피처
    "a_t_lag", "delta_a", "a_ma3", "a_trend", "a_b_ratio",
    
    # HS4 코드 관련 피처 (품목군 트렌드)
    "hs4_t", "hs4_ma3", "hs4_trend", "hs4_share",
    
    # 거래 특성 피처
    "w_t", "w_ma3", "unit_price",
    
    # 거래 건수 관련 피처
    "cnt_t", "cnt_ma3", "cnt_ratio_3",
    
    # 로그 변환 피처 (분포 정규화)
    "log_b_t", "log_a_t_lag",
    
    # 상관관계 피처 (품목 간 연관성)
    "max_corr", "best_lag",
    
    # 시간적 패턴 (월별 주기성)
    "month_sin", "month_cos", "month_a_inter",
    
    # 변동성 및 추세 피처
    "b_std6", "b_max_ratio", "b_min_ratio", "b_inc_cnt",
    
    # 연간 변화율 및 모멘텀
    "yoy_change", "a_lag2", "a_momentum", "b_ema",
]

# pair_score 추가 (품목 쌍의 상관관계 점수)
feature_cols_dl = feature_cols + ["pair_score"]

# Target 변수 생성: Log Difference 방식
df['log_current'] = np.log1p(df['b_t'])
df['target_diff'] = df['target'] - df['log_current']
y = df['target_diff'].astype("float32").values

# 데이터 정규화: QuantileTransformer로 정규분포 변환
scaler = QuantileTransformer(output_distribution='normal', random_state=42)
X_cont_all = df[feature_cols_dl].astype("float32").values
X_cont_test = test_df[feature_cols_dl].astype("float32").values

# Train/Valid 마스크 생성
train_mask = df["is_valid"] == 0
valid_mask = df["is_valid"] == 1

# Train 데이터로만 Scaler 학습 (데이터 누수 방지)
scaler.fit(X_cont_all[train_mask])
X_cont_all = scaler.transform(X_cont_all)
X_cont_test = scaler.transform(X_cont_test)

# 예측 복원을 위한 기준값 저장
base_test = test_df["b_t"].astype("float32").values


# ============================================================================
# PyTorch Dataset 정의
# ============================================================================
class TabularDataset(Dataset):
    """품목 인덱스 + 연속형 피처를 결합한 데이터셋"""
    
    def __init__(self, lead_idx, follow_idx, x_cont, y=None):
        self.lead_idx = torch.tensor(lead_idx, dtype=torch.long)
        self.follow_idx = torch.tensor(follow_idx, dtype=torch.long)
        self.x_cont = torch.tensor(x_cont, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None
    
    def __len__(self):
        return len(self.x_cont)
    
    def __getitem__(self, idx):
        if self.y is None:
            return self.lead_idx[idx], self.follow_idx[idx], self.x_cont[idx]
        return self.lead_idx[idx], self.follow_idx[idx], self.x_cont[idx], self.y[idx]


# ============================================================================
# FT-Transformer 모델 정의
# ============================================================================
class FTTransformer(nn.Module):
    """
    Feature Tokenizer Transformer
    - 각 피처를 토큰으로 변환 후 Transformer로 학습
    - CLS 토큰을 사용해 최종 예측 수행
    """
    
    def __init__(self, n_leads, n_follows, n_cont, 
                 d_embedding=48, n_layers=3, n_heads=8, dropout=0.2):
        super().__init__()
        
        # 품목 임베딩
        self.lead_emb = nn.Embedding(n_leads, d_embedding)
        self.follow_emb = nn.Embedding(n_follows, d_embedding)
        
        # 연속형 피처 임베딩
        self.cont_emb = nn.ModuleList([
            nn.Linear(1, d_embedding) for _ in range(n_cont)
        ])
        
        # CLS 토큰
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_embedding))
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_embedding,
            nhead=n_heads,
            dim_feedforward=int(d_embedding * 1.33),
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # 예측 헤드
        self.head = nn.Sequential(
            nn.LayerNorm(d_embedding),
            nn.ReLU(),
            nn.Linear(d_embedding, 1)
        )
    
    def forward(self, lead_idx, follow_idx, x_cont):
        bs = lead_idx.size(0)
        
        # 품목 임베딩
        l_emb = self.lead_emb(lead_idx).unsqueeze(1)
        f_emb = self.follow_emb(follow_idx).unsqueeze(1)
        
        # 연속형 피처 임베딩
        c_embs = [
            layer(x_cont[:, i].unsqueeze(1)).unsqueeze(1)
            for i, layer in enumerate(self.cont_emb)
        ]
        c_embs = torch.cat(c_embs, dim=1)
        
        # 모든 토큰 결합
        x = torch.cat([
            self.cls_token.expand(bs, -1, -1),
            l_emb,
            f_emb,
            c_embs
        ], dim=1)
        
        # Transformer 처리
        x = self.transformer(x)
        
        # CLS 토큰으로 예측
        return self.head(x[:, 0, :]).squeeze(-1)


# ============================================================================
# 데이터 로더 생성
# ============================================================================
train_ds = TabularDataset(
    df.loc[train_mask, 'lead_idx'].values,
    df.loc[train_mask, 'follow_idx'].values,
    X_cont_all[train_mask],
    y[train_mask]
)

valid_ds = TabularDataset(
    df.loc[valid_mask, 'lead_idx'].values,
    df.loc[valid_mask, 'follow_idx'].values,
    X_cont_all[valid_mask],
    y[valid_mask]
)

test_ds = TabularDataset(
    test_df['lead_idx'].values,
    test_df['follow_idx'].values,
    X_cont_test,
    None
)

# Windows에서는 num_workers=0, Linux/Mac은 2
num_workers = 0 if os.name == 'nt' else 2

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=num_workers)
valid_loader = DataLoader(valid_ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=num_workers)


# ============================================================================
# FT-Transformer 학습 (3-Seed 앙상블)
# ============================================================================
ft_preds_test_list = []
ft_models_state = []

print("\n🚀 [1/2] Training FT-Transformer (3-Seed Ensemble)...")

for seed in cfg.SEEDS:
    print(f"\n📌 Training with Seed {seed}...")
    seed_everything(seed)
    
    # 모델 초기화
    model = FTTransformer(n_leads, n_follows, X_cont_all.shape[1]).to(cfg.DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.LR,
        weight_decay=cfg.WEIGHT_DECAY
    )
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    criterion = nn.MSELoss()
    
    best_loss = float('inf')
    best_state = None
    patience = 0
    
    # 학습 루프
    for epoch in range(1, cfg.EPOCHS + 1):
        model.train()
        for l, f, x, target in train_loader:
            l, f, x, target = (
                l.to(cfg.DEVICE),
                f.to(cfg.DEVICE),
                x.to(cfg.DEVICE),
                target.to(cfg.DEVICE)
            )
            
            optimizer.zero_grad()
            loss = criterion(model(l, f, x), target)
            loss.backward()
            optimizer.step()
        
        scheduler.step()
        
        # 검증
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for l, f, x, target in valid_loader:
                l, f, x, target = (
                    l.to(cfg.DEVICE),
                    f.to(cfg.DEVICE),
                    x.to(cfg.DEVICE),
                    target.to(cfg.DEVICE)
                )
                val_loss += criterion(model(l, f, x), target).item()
        
        # Early Stopping
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = model.state_dict()
            patience = 0
        else:
            patience += 1
            if patience >= 10:
                print(f"⏹️  Early stopping at epoch {epoch}")
                break
    
    # 최적 모델 저장
    ft_models_state.append(best_state)
    
    # 테스트 예측
    model.load_state_dict(best_state)
    model.eval()
    temp_preds = []
    
    with torch.no_grad():
        for l, f, x in test_loader:
            l, f, x = l.to(cfg.DEVICE), f.to(cfg.DEVICE), x.to(cfg.DEVICE)
            temp_preds.append(model(l, f, x).cpu().numpy())
    
    # Log Difference → 실제 값 복원
    pred_diff = np.concatenate(temp_preds)
    pred_raw = np.expm1(pred_diff + np.log1p(base_test))
    ft_preds_test_list.append(pred_raw)

# 3개 시드 예측값 평균
ft_final_pred = np.mean(ft_preds_test_list, axis=0)
print("✅ FT-Transformer training complete!")


# ============================================================================
# LightGBM 학습 (Deep Tuning)
# ============================================================================
print("\n🌲 [2/2] Training LightGBM (Deep Tuned)...")

X_lgb_train = df.loc[train_mask, feature_cols_dl]
y_lgb_train = df.loc[train_mask, "target_diff"]
X_lgb_valid = df.loc[valid_mask, feature_cols_dl]
y_lgb_valid = df.loc[valid_mask, "target_diff"]
X_lgb_test = test_df[feature_cols_dl]

lgb_train = lgb.Dataset(X_lgb_train, label=y_lgb_train)
lgb_valid = lgb.Dataset(X_lgb_valid, label=y_lgb_valid, reference=lgb_train)

# 초저학습률 + 높은 반복수로 세밀하게 학습
lgb_params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'learning_rate': 0.005,
    'num_leaves': 64,
    'colsample_bytree': 0.7,
    'subsample': 0.7,
    'subsample_freq': 1,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_jobs': -1,
    'seed': 42,
    'verbose': -1
}

model_lgb = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=[lgb_valid],
    callbacks=[
        lgb.early_stopping(stopping_rounds=500),
        lgb.log_evaluation(1000)
    ]
)

# 예측 및 복원
lgb_pred_diff = model_lgb.predict(X_lgb_test)
lgb_final_pred = np.expm1(lgb_pred_diff + np.log1p(base_test))
lgb_final_pred = np.maximum(0, lgb_final_pred)

print("✅ LightGBM training complete!")


# ============================================================================
# 💾 모델 가중치 저장
# ============================================================================
print(f"\n💾 Saving model weights to {cfg.OUTPUT_DIR}...")

# 1. FT-Transformer 전체 앙상블 저장
torch.save({
    'model_states': ft_models_state,
    'n_leads': n_leads,
    'n_follows': n_follows,
    'n_cont': X_cont_all.shape[1],
    'feature_cols_dl': feature_cols_dl,
    'config': {
        'd_embedding': 48,
        'n_layers': 3,
        'n_heads': 8,
        'dropout': 0.2
    }
}, os.path.join(cfg.OUTPUT_DIR, 'ft_transformer_ensemble.pth'))

# 2. 전처리 객체 저장
with open(os.path.join(cfg.OUTPUT_DIR, 'preprocessors.pkl'), 'wb') as f:
    pickle.dump({
        'scaler': scaler,
        'le_lead': le_lead,
        'le_follow': le_follow
    }, f)

# 3. LightGBM 모델 저장
model_lgb.save_model(os.path.join(cfg.OUTPUT_DIR, 'lightgbm_model.txt'))

print("✅ Models saved successfully!")


# ============================================================================
# 앙상블 및 Smart Filtering
# ============================================================================
print("\n⚖️  Final Ensemble (FT 55% + LGB 45%)...")

# 앙상블
final_pred = (ft_final_pred * 0.55) + (lgb_final_pred * 0.45)
final_pred = np.maximum(0, final_pred)

# Smart Filter 적용
print("\n🎯 Applying Smart Filter (Based on pair_score)...")

submission = test_df.copy()
submission["value"] = final_pred

# pair_score 기반 하위 5% 제거
score_threshold = np.percentile(submission['pair_score'], 5)
print(f"📉 Pair Score Threshold: {score_threshold:.4f} (Removing bottom 5%)")

original_len = len(submission)
submission_filtered = submission[submission['pair_score'] > score_threshold].copy()
new_len = len(submission_filtered)

print(f"🗑️  Rows dropped: {original_len} → {new_len} (-{original_len - new_len} rows)")

# 최종 제출 파일 생성
final_sub = submission_filtered[["leading_item_id", "following_item_id", "value"]]
final_sub = final_sub.groupby(
    ["leading_item_id", "following_item_id"],
    as_index=False
)["value"].sum()

save_path = os.path.join(cfg.OUTPUT_DIR, "submission_smart_filter.csv")
final_sub.to_csv(save_path, index=False)

print(f"\n✅ Submission file created: {save_path}")

print("\n" + "="*70)
print("🎉 전체 작업 완료!")
print("="*70)
print(f"\n📦 생성된 파일 (in {cfg.OUTPUT_DIR}/):")
print("   1. submission_smart_filter.csv (제출용)")
print("   2. ft_transformer_ensemble.pth (모델 가중치)")
print("   3. preprocessors.pkl (전처리 객체)")
print("   4. lightgbm_model.txt (LightGBM 모델)")

📍 현재 파이썬 실행 위치: c:\Users\User\Desktop\데이콘 제출_헬퍼_장광남\Finals
✅ 상위 폴더의 'data' 폴더에서 파일을 찾았습니다.
📂 Loading data from c:\Users\User\Desktop\데이콘 제출_헬퍼_장광남\data...
✅ Data loaded successfully!
   Train shape: (47897, 44)
   Test shape: (1764, 41)

🚀 [1/2] Training FT-Transformer (3-Seed Ensemble)...

📌 Training with Seed 41...


KeyboardInterrupt: 